# 1. Historical versus corrected nonlinear averaging

This isolates the mathematical averaging change at the same surface centre and footprint size. The diagonal represents identical results.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import footprint_tools as ft
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}

data = ft.load_cache()
filled = data[(data.footprint_kind == "Filled") & (data.pv_averaging == "nonlinear")].copy()
score = ft.footprint_scorecard(data)


In [ ]:
fig, axes = plt.subplots(2,2, figsize=(11,9), constrained_layout=True)
for row, frac in enumerate([0.5, 1.0]):
    old = data[data.footprint.eq(f"legacy_{frac:g}")]
    new = data[data.footprint.eq(f"filled_{frac:g}")]
    paired = old.merge(new, on=["Eddy","Day"], suffixes=("_legacy","_new"), validate="one_to_one")
    for ax, column, label in [(axes[row,0], "PV_grad_topo_mag", "Topographic net magnitude"),
                              (axes[row,1], "topo_plan_ratio", "log topo/planetary")]:
        x, y = paired[f"{column}_legacy"], paired[f"{column}_new"]
        positive = (x > 0) & (y > 0) if column.endswith("mag") else np.isfinite(x+y)
        ax.hexbin(x[positive], y[positive], gridsize=40, mincnt=1, bins="log", cmap="magma")
        lo, hi = np.nanpercentile(np.r_[x[positive],y[positive]], [1,99])
        ax.plot([lo,hi],[lo,hi],"w--",lw=1); ax.set(xlabel="Legacy", ylabel="Corrected", title=f"frac={frac:g}: {label}")
        if column.endswith("mag"): ax.set(xscale="log", yscale="log")
plt.show()

In [ ]:
pair = data[data.footprint.isin(["legacy_1","filled_1"])].pivot(index=["Eddy","Day","Cyc"], columns="footprint", values="preference_error_deg").dropna().reset_index()
pair["error_change"] = pair.filled_1 - pair.legacy_1
fig, ax = plt.subplots(figsize=(9,4.5), constrained_layout=True)
sns.violinplot(data=pair, x="Cyc", y="error_change", palette=palette, hue="Cyc", legend=False, inner="quart", ax=ax)
ax.axhline(0,color="black",lw=.8); ax.set(ylabel="Corrected minus legacy tilt-direction error (degrees)", title="Does nonlinear averaging change directional agreement?")
plt.show()